# Jämförelse av återbetalningsplaner för studielån med PROC LOAN

## Sammanfattning

Ett kontor för studiefinansiering utvärderar hur en avgångskull bör återbetala en representativ skuld på 27 500 dollar genom att jämföra fyra återbetalningsstrukturer — en federal lån med fast ränta, en privat refinansiering med uppläggningsavgift, ett räntetaksbelagt rörligt lån (ARM) och en arbetsgivarsubventionerad räntenedsättning — med hjälp av `PROC LOAN`. Över en löptid på 120 månader ligger de fyra erbjudandena nära varandra i månadsbetalning och total ränta vid sina angivna starträntor: den federala standardplanen kostar mest i ränta (**10 021,22 dollar** vid 6,53 %, betalning **312,68 dollar**), den privata refinansieringen sänker räntan men lägger till en avgift på 412,50 dollar (**9 120,20 dollar** vid 5,99 %, betalning **305,17 dollar**), och den lägre angivna ARM-räntan (**7 099,76 dollar** vid 4,75 %, betalning **288,33 dollar**) samt arbetsgivarens räntenedsättning (**6 700,67 dollar** vid 4,50 %, betalning **285,01 dollar**) medför de lägsta räntekostnaderna. Ett `COMPARE`-block rapporterar sedan varje plans ackumulerade ränta, amorterat kapital och utestående saldo vid 36, 60 och 120 månader, vilket visar hur långt varje lån har amorterats vid de tidpunkter en låntagare oftast överväger refinansiering eller full återbetalning.

## Datakällor

| Dataset | Rader | Beskrivning | Nyckelvariabler |
|---------|------|-------------|---------------|
| `borrowers` | 40 | Syntetiska lånprofiler för en avgångskull, genererade inline med `call streaminit(20260531)` och `rand('uniform')`. Används för att motivera realistiska lånevillkor inför jämförelsen. | `student_id` (1001-1040), `balance` (kapitalbelopp vid examen; detta urval spänner 11 800-47 750 dollar, medelvärde 30 771 dollar), `apr` (4,10 %-9,10 % nominell årsränta, medelvärde 6,50 %), `term` (120 eller 180 månader, medelvärde 144), `origination_pct` (1,0 %-2,0 % avgift, medelvärde 1,50 %) |

Den representativa låntagaren som analyseras med `PROC LOAN` (belopp 27 500 dollar, löptid 120 månader, start juli 2026) ligger nära den nedre mittendelen av denna kohorts skuld- och räntefördelning; ingen extern data eller nätverksdata används. Kohorten finns för att motivera rimliga lånevillkor — jämförelsen görs på det enskilda representativa lånet.

# Jämförelse av återbetalningsplaner för studielån med PROC LOAN

När studenter tar examen måste ett kontor för studiefinansiering hjälpa dem att välja mellan konkurrerande återbetalningserbjudanden. `PROC LOAN` (SAS/ETS) är skräddarsytt för exakt detta: det amorterar lån med fast ränta, rörlig ränta (ARM) och räntenedsättning ("buydown") och jämför dem sedan sida vid sida på betalning, total ränta och amorteringsförlopp.

I den här notebooken kommer vi att:

1. Generera en syntetisk avgångskull för att fastställa realistiska lånevillkor.
2. Sammanfatta kohorten med `PROC MEANS`.
3. Bygga fyra återbetalningsplaner för ett representativt saldo på 27 500 dollar och jämföra dem med `PROC LOAN` (FIXED / ARM / BUYDOWN + COMPARE).
4. Köra om den rekommenderade fasta planen separat för att bekräfta dess fristående ekonomi.

Allt körs lokalt på inline-genererad syntetisk data — inga externa filer eller nätverksåtkomst behövs.

## Steg 1 — Generera en syntetisk avgångskull

Vi simulerar 40 låntagare. Var och en får ett kapitalbelopp vid examen, en årsränta löst kopplad till kreditkvalitet, en standardåterbetalningstid (10 eller 15 år) och en uppläggningsavgift i procent. Fröet gör datan reproducerbar.

In [1]:
data borrowers;
   CALL streaminit(20260531);
   LÄNGD plan $ 28;
   GÖR student_id = 1001 TILL 1040;
      /* Kapitalbelopp vid examen: 9 500 - 48 000 dollar */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* Nominell årsränta efter kreditklass: 4,0 % - 9,5 % */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* Standardåterbetalningstid: 120 eller 180 månader */
      OM rand('uniform') < 0.6 SÅ term = 120;
      ANNARS term = 180;
      /* Uppläggningsavgift i procent av kapitalet: 1,0 % - 2,0 % */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      UTDATA;
   SLUT;
KÖR;


NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Steg 2 — Profilera kohorten

Innan vi modellerar enskilda lån tittar vi på fördelningen av saldon, räntor och löptider. Detta visar hur en *representativ* låntagare ser ut — grunden för den direktjämförelse som följer.

In [2]:
PROCEDUR MEDELVÄRDEN data=borrowers n mean MIN MAX maxdec=2;
   VARIABEL balance apr term origination_pct;
KÖR;

                                                  The MEANS Procedure

 Variable               N           Mean     Minimum     Maximum
 ---------------------------------------------------------------
 balance               40       30771.25    11800.00    47750.00
 apr                   40           6.50        4.10        9.10
 term                  40         144.00      120.00      180.00
 origination_pct       40           1.50        1.00        2.00
 ---------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Steg 3 — Jämför fyra återbetalningsplaner för en representativ låntagare

Med ett representativt saldo på 27 500 dollar över en löptid på 120 månader (10 år) med start i juli 2026 ställer vi upp fyra realistiska erbjudanden:

- **Federalt direktlån, ej subventionerat (Standard)** — ett enkelt lån med fast ränta på 6,53 %.
- **Privat refinansiering (med avgift)** — en lägre fast ränta på 5,99 %, men med en uppläggningskostnad på 412,50 dollar (`INIT=`).
- **Privat rörlig ränta (ARM)** — ett justerbart lån på 4,75 % med 1 % per justering / 5 % livstidstak (`CAPS=`), en maxränta på 9,75 % (`MAXRATE=`), årlig justeringsfrekvens (`ADJUSTFREQ=`) och nyckelordet `WORSTCASE`.
- **Arbetsgivarens 2-1-räntenedsättning** — en subventionerad start på 4,50 % som enligt det angivna schemat trappas upp via `BDRATES=` till 5,50 % år 2 och 6,50 % därefter.

`COMPARE`-satsen begär den lånövergripande vyn vid 36, 60 och 120 månader, med en skattesats (`TAXRATE=`) på 22 % och en lägsta attraktiva avkastning (`MARR=`) på 4 %; `OUTSUM=` och `OUTCOMP=` fångar sammanfattningen per lån respektive jämförelseraderna. Listningen nedan redovisar, för varje plan och varje kontrollpunkt, den **ackumulerade betalda räntan, det ackumulerade amorterade kapitalet och det utestående saldot** — en tydlig bild av amorteringsförloppet mellan kandidaterna.

> **Notera om räntejusteringar.** Jenners `PROC LOAN` amorterar varje plan till dess angivna **start**ränta, så ARM-lånets `CAPS`/`MAXRATE`/`WORSTCASE`-inställningar och räntenedsättningens `BDRATES`-steg redovisas i listningen som lånets villkor men **tillämpas inte** på betalningsberäkningen — ARM- och räntenedsättningssiffrorna nedan speglar deras starträntor på 4,75 % respektive 4,50 % hållna konstanta över hela löptiden. Betrakta dessa två totalsummor som bästa-scenario-kostnader (startränta), inte värsta-scenario-tak.

In [3]:
PROCEDUR loan START=2026:7 amount=27500 life=120 outsum=plan_sum;

   fixed rate=6.53
         label="Federalt lån, Standard";

   fixed rate=5.99 init=412.50
         label="Privat refinansiering";

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       label="ARM (rörlig ränta)";

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           label="Arbetsgivarrabatt 2-1";

   COMPARE at=(36 60 120) ALL taxrate=22 marr=4 OUTCOMP=plan_cmp;
KÖR;

                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: Federalt lån, Standard
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: Privat refinansiering
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:                 412.50

  Loan #3: ARM (rörlig ränta)
    Loan Type:                   Adjustable Rate
    Amount (Principal):                27500.00
    Interest Rate (


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Steg 4 — Kör om den rekommenderade fasta planen separat

För den låntagare som värdesätter förutsägbar betalning är den federala standardplanen med fast ränta det säkra förvalet. Vi kör om den separat för att bekräfta dess huvudsakliga ekonomi: samma **312,68 dollar** i månadsbetalning, **37 521,22 dollar** totalt betalt och **10 021,22 dollar** total ränta som sågs i fyrvägsjämförelsen, nu presenterat som en fristående lånesammanfattning.

In [4]:
PROCEDUR loan START=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed label="Federalt lån, Standard";
KÖR;

                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: Federalt lån, Standard
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan Repayment Schedule: Federalt lån, Standard
      Year    Begin Balance        Payment       Interest      Principal      End Balance
    ------ ---------------- -------------- -------------- -------------- ----------------
         1         27500.00        3752.12        1736.12        2016.00         25484.00
         2         25484.00        3752.12        1600.47        2151.66         23332.34
         3         23332.34        3752.12        1455.68        2296.44         21035.90
         4         21035.90        3752.12        1301.


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Tolkning av resultaten

Alla fyra planer amorterar fullt ut samma kapitalbelopp på 27 500 dollar över 120 månader (varje plans utestående saldo når 0,00 dollar vid månad 120 i COMPARE-tabellen), så beslutet avgörs av **månadsbetalning** och **total ränta vid angiven ränta**:

| Plan | Ränta | Betalning | Total ränta | Anmärkningar |
|------|------|---------|----------------|-------|
| Federalt direktlån, Standard | 6,53 % | 312,68 dollar | 10 021,22 dollar | Ingen avgift; starkast låntagarskydd |
| Privat refinansiering | 5,99 % | 305,17 dollar | 9 120,20 dollar | Uppläggningsavgift på 412,50 dollar |
| Privat rörlig ränta (ARM) | 4,75 % (start) | 288,33 dollar | 7 099,76 dollar | Räntan kan stiga; siffran är startränte-kostnaden |
| Arbetsgivarens 2-1-räntenedsättning | 4,50 % (start) | 285,01 dollar | 6 700,67 dollar | Beror på fortsatt anställning |

- Den **federala standardplanen** är dyrast i ränta (10 021,22 dollar) men erbjuder en fast, förutsägbar betalning på 312,68 dollar utan avgift.
- Den **privata refinansieringen** sänker räntan till 5,99 % (9 120,20 dollar i ränta, 901 dollar mindre än den federala planen) men tar ut en uppläggningsavgift på 412,50 dollar, så nettofördelen jämfört med den federala planen är ungefär 488 dollar i ränta minus avgift — meningsfullt bara om lånet löper tillräckligt länge för att inte refinansieras bort.
- **ARM-lånet** och **räntenedsättningen** visar den lägsta räntan här (7 099,76 dollar respektive 6 700,67 dollar) enbart för att deras **start**räntor ligger klart under de fasta erbjudandena. Som noterades i steg 3 håller Jenner dessa starträntor konstanta, så dessa är bästa-scenario-siffror: en verklig ARM som justeras upp — eller en räntenedsättning som förlorar sin arbetsgivarsubvention — skulle hamna högre, och betalningen är inte garanterad.

`COMPARE`-tabellen förtydligar detta genom att visa hur snabbt varje plan amorteras. Vid 36 månader har den federala planen betalat 4 792,27 dollar i ränta och amorterat 6 464,10 dollar av kapitalet (saldo 21 035,90 dollar), medan räntenedsättningen bara har betalat 3 263,97 dollar i ränta men amorterat 6 996,24 dollar av kapitalet (saldo 20 503,76 dollar) — planerna med lägre ränta kostar både mindre i ränta och amorterar kapitalet snabbare under de första tre åren. För en riskavert nyexaminerad motiverar den federala standardplanens räntesäkerhet ofta dess högre ränta; för en låntagare som är trygg i en stabil, varaktig anställning ger räntenedsättningens subventionerade start de största besparingarna bland de alternativ som faktiskt låser räntan.